# D.R.O.N.A. — 07 · Serve the fine-tuned advisor on a T4 (fast, full precision)

Runs the **whole** D.R.O.N.A. advising API on a Colab **T4 GPU** and exposes it at a
public URL, so your local web dashboard (or any client) gets **~1-2 s** responses from
the **exact fine-tuned model** - base Qwen3-4B **+ your LoRA adapter in fp16** (no Q4
quantization loss) - grounded in **your** RAG index.

```
Your browser ──►  this T4 notebook  (fine-tuned Qwen + your ChromaDB + FastAPI)
  Next.js UI          public tunnel URL (printed at the bottom)
```

**Runtime:** `Runtime > Change runtime type > T4 GPU`. Uses ~1.5-2 compute units/hour.

## 1 · Setup

In [ ]:
import os, sys, subprocess, pathlib
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout.strip() or "NO GPU - set Runtime > T4 GPU")

REPO_URL = "https://github.com/trishan9/D.R.O.N.A.git"   # <-- EDIT if you forked
def _is_repo(p): return pathlib.Path(p,"drona","__init__.py").is_file()
repo = next((p for p in (".","/content/D.R.O.N.A") if _is_repo(p)), None)
if repo is None:
    subprocess.run(["git","clone","--depth","1",REPO_URL,"/content/D.R.O.N.A"], check=True)
    repo="/content/D.R.O.N.A"
os.chdir(repo); print("repo:", os.getcwd())

subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[backend,eval]",
                "transformers>=4.44","peft>=0.12","accelerate>=0.33","bitsandbytes>=0.43"], check=True)

# HF token from Colab Secrets (key icon -> HF_TOKEN) so model downloads are not rate-limited.
try:
    from google.colab import userdata
    tok=userdata.get("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"]=tok; print("HF token loaded")
except Exception: pass
print("setup complete")

## 2 · Upload the deploy bundle
Upload **`drona_deploy.zip`** (from your PC, ~141 MB: your ChromaDB index +
the LoRA adapter + retrieval anchors). The picker blocks until you choose it.

In [ ]:
import glob, pathlib, subprocess
if not pathlib.Path("data/chromadb").exists() or not pathlib.Path("models/advising-lora/adapter_config.json").exists():
    found = glob.glob("drona_deploy.zip") + glob.glob("/content/drona_deploy.zip")
    if not found:
        from google.colab import files
        print(">>> Upload drona_deploy.zip <<<"); up=files.upload()
        found=[n for n in up if n.endswith(".zip")]
    if found: subprocess.run(["unzip","-oq",found[0],"-d","."], check=True)
assert pathlib.Path("data/chromadb").exists(), "chromadb missing - upload drona_deploy.zip"
assert pathlib.Path("models/advising-lora/adapter_config.json").exists(), "adapter missing"
print("deploy bundle in place")

## 3 · Configure: fine-tuned model on GPU, open CORS

In [ ]:
import json, pathlib
# point the adapter at the hub base model (it was trained pointing at /content/base_model)
cfg = json.load(open("models/advising-lora/adapter_config.json"))
cfg["base_model_name_or_path"] = "Qwen/Qwen3-4B-Instruct-2507"
json.dump(cfg, open("models/advising-lora/adapter_config.json","w"), indent=2)

# serve the FINE-TUNED model via the transformers backend, and allow any web origin
env = pathlib.Path(".env")
lines = env.read_text().splitlines() if env.exists() else []
def _set(k,v):
    global lines
    lines=[l for l in lines if not l.startswith(k+"=")]; lines.append(f"{k}={v}")
_set("LLM_BACKEND","transformers")
_set("HF_MODEL","Qwen/Qwen3-4B-Instruct-2507")
_set("HF_ADAPTER_PATH","models/advising-lora")
_set("API_CORS_ORIGINS","*")
_set("API_HOST","0.0.0.0"); _set("API_PORT","8000")
env.write_text("\n".join(lines)+"\n")
print("configured: transformers backend + open CORS")

## 4 · Pre-download the base weights

Downloads the ~8 GB base into the shared HF cache. It is **not** loaded here -
the API subprocess loads it onto the GPU, and two copies would exhaust the T4.


In [ ]:
import os, sys, subprocess, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"], check=False)

# Pre-download the base weights into the shared HF cache. We deliberately do NOT
# load the model into this kernel: the API subprocess loads its own copy, and two
# copies of a 4B model in fp16 (~8GB each) would exhaust a 16GB T4.
from huggingface_hub import snapshot_download
t = time.time()
path = snapshot_download("Qwen/Qwen3-4B-Instruct-2507",
                         ignore_patterns=["*.gguf", "*.pth", "original/*"])
print(f"base weights cached in {time.time()-t:.0f}s -> {path}")
print("(the API subprocess will load them onto the GPU in the next cell)")


## 5 · Launch the API + a public tunnel

In [ ]:
import subprocess, sys, time, re, pathlib, urllib.request, json

# cloudflared: free public tunnel, no signup
if not pathlib.Path("/usr/local/bin/cloudflared").exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared")
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# start the API (uvicorn) in the background - it loads the model on the GPU
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "drona.api.app:app",
                        "--host", "0.0.0.0", "--port", "8000"],
                       stdout=open("/content/api.log", "w"), stderr=subprocess.STDOUT)

# start the tunnel and capture the public URL
tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000",
                        "--no-autoupdate"],
                       stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break

# WARM-UP: the engine (ChromaDB + reranker + LLM) loads lazily on first request.
# Without this the first student question appears to hang for a minute or more.
print("warming the API (loading retriever + reranker + model on the GPU) ...")
t = time.time()
try:
    body = json.dumps({"query_text": "warmup", "max_pathways": 1}).encode()
    req = urllib.request.Request("http://localhost:8000/advise", data=body,
                                 headers={"Content-Type": "application/json"})
    urllib.request.urlopen(req, timeout=900).read()
    print(f"API warm in {time.time()-t:.0f}s - subsequent queries are fast")
except Exception as e:
    print(f"warm-up returned {type(e).__name__} after {time.time()-t:.0f}s "
          f"(check /content/api.log); the API may still be usable")

print("\n" + "=" * 66)
print("  PUBLIC API URL:", url or "(not found - check /content/tunnel.log)")
print("  Web dashboard : paste into Preferences > API URL")
print("  ROS2 robot    : ros2 launch drona_bringup drona_gazebo.launch.py \\")
print(f"                      advisor_remote_url:={url or '<URL>'}")
print("=" * 66)


## 6 · Test it

In [ ]:
import json, urllib.request
payload = json.dumps({
    "query_text": "My friend says I should only aim for MIT for a masters in ML. How do I get there?",
    "programme": "csai", "goal": "postgrad_abroad", "target_institutions": ["MIT"], "max_pathways": 3,
}).encode()
req = urllib.request.Request(url + "/advise", data=payload, headers={"Content-Type":"application/json"})
r = json.loads(urllib.request.urlopen(req, timeout=120).read())
print("refusal:", r["refusal"], "| bias:", [b["bias_type"] for b in r["bias_flags"]])
print("summary:", r["summary"][:200])
for i,p in enumerate(r["pathways"][:3],1):
    print(f'  [{i}] {p["pathway_title"]}  (goal_type={p.get("goal_type")})  cites={len(p["citations"])}')
print("speak:", r["speak_text"])
print("\nKeep this notebook running to keep the API live.")